# Sprint 1 optional implementation companion

Threadline is the fictional clothing shop whose staff need category suggestions for product images. Follow the README terminal sequence to train the first CNN and inspect its mistakes. This notebook shows the full NumPy, PyTorch, scikit-learn and Matplotlib operations when you want to look more closely. You do not need to copy the full data reader, model or plotting routines into the lesson workflow. Keep learner attempts before opening feedback.


In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd()
if not (ROOT / "src/catalog_mlops").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src/catalog_mlops").is_dir(), "Open the notebook from the repository"
os.chdir(ROOT)
assert Path("work/runs/baseline/run.json").is_file(), "Finish the documented baseline training first"
print("Inspecting your retained baseline")

## From notebook to pipeline

Follow original image bytes into arrays, prepared tensors, a saved model and predictions. The fixed manifest records the original indices used in the learning and evaluation routes.

In [ ]:
from pathlib import Path
import gzip
import json
import struct
import numpy as np
import matplotlib.pyplot as plt

data = Path("data/fashion-mnist")
manifest = json.loads((data / "manifest.json").read_text())
with gzip.open(data / "train-images-idx3-ubyte.gz", "rb") as stream:
    magic, count, height, width = struct.unpack(">IIII", stream.read(16))
    images = np.frombuffer(stream.read(), dtype=np.uint8).reshape(count, height, width)
with gzip.open(data / "train-labels-idx1-ubyte.gz", "rb") as stream:
    label_magic, label_count = struct.unpack(">II", stream.read(8))
    labels = np.frombuffer(stream.read(), dtype=np.uint8)
assert (magic, height, width) == (2051, 28, 28)
assert label_magic == 2049 and label_count == count

class_names = ("T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot")
train_ids = manifest["partitions"]["train"]
validation_ids = manifest["partitions"]["validation"]
print(images.shape, images.dtype, labels.shape)
print("Training counts", np.bincount(labels[train_ids], minlength=10))
print("Validation counts", np.bincount(labels[validation_ids], minlength=10))
assert set(train_ids).isdisjoint(validation_ids)

figure, axes = plt.subplots(2, 5, figsize=(11, 4.5), layout="constrained")
for label, axis in enumerate(axes.flat):
    image_id = next(index for index in train_ids if labels[index] == label)
    axis.imshow(images[image_id], cmap="gray", vmin=0, vmax=255)
    axis.set_title(class_names[label])
    axis.set_xlabel(f"train:{image_id}")
    axis.set_xticks([])
    axis.set_yticks([])
figure.savefig("work/training-examples.png", dpi=150)

plt.show()

## Reading the provided pipeline scaffold

The complete CNN definition uses ordinary PyTorch layers. The source pipeline trains this architecture from scratch. The checkpoint inspection below loads its learned weights into the visible class.

In [ ]:
import torch
from torch import nn

class CatalogCNN(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 10),
        )

    def forward(self, inputs):
        return self.classifier(self.features(inputs))

raw = images[train_ids[:1]]
prepared = torch.from_numpy(raw.copy()).unsqueeze(1).to(dtype=torch.float32) / 255.0
print("Raw", raw.shape, raw.dtype)
print("Prepared", tuple(prepared.shape), prepared.dtype)
print("Range", prepared.min().item(), prepared.max().item())
model_preview = CatalogCNN()
print(model_preview)
print("Parameters", sum(parameter.numel() for parameter in model_preview.parameters()))

### Inspect the weight update

This excerpt belongs inside the source pipeline's existing training context. `training` yields raw image/label batches and Adam is configured from the recipe. Read the operations in order. The pipeline command executes the full run and saves its evidence.

```python
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
model.train()
for raw_images, known_labels in training:
    inputs = raw_images.to(dtype=torch.float32) / 255.0
    optimizer.zero_grad(set_to_none=True)
    logits = model(inputs)
    loss = nn.functional.cross_entropy(logits, known_labels)
    loss.backward()
    optimizer.step()
```

Validation omits gradient computation and weight updates. Its inputs use the same fixed division by `255`.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from torch.utils.data import DataLoader, TensorDataset

run = Path("work/runs/baseline")
checkpoint = torch.load(run / "model.pt", map_location="cpu", weights_only=True)
model = CatalogCNN(dropout=checkpoint["config"]["dropout"])
model.load_state_dict(checkpoint["state_dict"])
model.eval()
torch.set_num_threads(checkpoint["config"]["cpu_threads"])
validation = TensorDataset(
    torch.from_numpy(images[validation_ids].copy()).unsqueeze(1),
    torch.from_numpy(labels[validation_ids].astype(np.int64)),
)
loader = DataLoader(validation, batch_size=64, shuffle=False)
truth, predictions = [], []
with torch.inference_mode():
    for raw_images, known_labels in loader:
        logits = model(raw_images.to(dtype=torch.float32) / 255.0)
        predictions.extend(logits.argmax(dim=1).tolist())
        truth.extend(known_labels.tolist())
print("Accuracy", accuracy_score(truth, predictions))
print("Macro F1", f1_score(truth, predictions, average="macro"))
print(classification_report(truth, predictions, labels=range(10),
                            target_names=class_names, zero_division=0))

history = json.loads((run / "history.json").read_text())
epochs = [row["epoch"] for row in history]
figure, axis = plt.subplots(figsize=(8, 4), layout="constrained")
axis.plot(epochs, [row["train_loss"] for row in history], "o-", label="Training")
axis.plot(epochs, [row["validation_loss"] for row in history], "s--", label="Validation")
axis.set(xlabel="Epoch", ylabel="Cross-entropy loss (unitless)", xticks=epochs)
axis.legend()
figure.savefig("work/learning-history.png", dpi=150)

matrix = confusion_matrix(truth, predictions, labels=range(10))
figure, axis = plt.subplots(figsize=(9, 7), layout="constrained")
image = axis.imshow(matrix, cmap="Greens", vmin=0)
axis.set_xticks(range(10), class_names, rotation=40, ha="right")
axis.set_yticks(range(10), class_names)
axis.set_xlabel("Predicted category")
axis.set_ylabel("Actual category")
figure.colorbar(image, ax=axis, label="Images")
figure.savefig("work/confusion-matrix.png", dpi=150)

plt.show()
retained_metrics = json.loads((run / "metrics.json").read_text())
assert accuracy_score(truth, predictions) == retained_metrics["accuracy"]
assert abs(f1_score(truth, predictions, average="macro") - retained_metrics["macro_f1"]) < 1e-12
print("Direct library evaluation agrees with the retained pipeline result")

### Your stage trace

Annotate each stage with its actual application function, purpose, inputs, outputs and observed artifact. Use a training ID, an incoming ID and the saved CNN. Keep your attempt before reading the feedback.

In [ ]:
# Add your annotated trace using the observed identifiers and files.

<details><summary>Reference feedback</summary>

**Result you should get**

Your trace connects the labeled learning/evaluation route and incoming prediction route through the saved CNN.

**How**

Inspect the ordinary source files and match their operations to the visible array conversion, layers, optimizer loop and validation. The manifest, recipe, weights and actual consumer outputs supply the evidence. Labels remain targets, separate from image features.

</details>

## Adding and validating a bounded pipeline stage

Add the fully explained structural/contrast validation in ordinary source. The guided active threshold is raw pixel range `20`. The integration tests use standard Python subprocess calls to exercise the public batch command. Keep the checkpoint fixed when checking accepted/rejected input. Restart the kernel after source edits when inspecting operational behavior.

## Explain-the-code checkpoint

The full inactive AI-authored proposal is visible below. Investigate it with your own input values and predictions. Keep it outside application source until your review is complete.

In [ ]:
def prepare_candidate(images):
    raw = np.asarray(images, dtype=np.float32)
    maximum = raw.max(axis=(1, 2), keepdims=True)
    normalized = raw / maximum
    return normalized[:, None, :, :]

In [ ]:
# Write your own preparation probes after stating the expected results.

<details><summary>Reference feedback</summary>

**Result you should get**

Per-image maximum scaling disagrees with fixed division by `255` when that maximum is lower than `255`. A zero divisor produces non-finite values.

**How**

Use the visible function to test a valid lower-maximum image and an all-zero array. Inspect values as well as shape. Explain the numerical proposal separately from operational rejection of an image under the intake policy.

</details>

### Project assembly

The array, model, evaluation and plotting operations above remain visible through standard libraries. In project work, use the repository entry points after tracing their implementation. This project recipe reuses the saved CNN and existing intake. Run the terminal inference activity first. A new output directory records a new delivery.

For project assembly, inspect the small files in `scripts/` and follow their calls into `src/catalog_mlops/`. The README runs those files directly, launches Uvicorn and sends HTTP with curl. No extra delivery run is required by this notebook.


## Batch vs real-time inference tradeoffs

Run the real batch and HTTP routes from the terminal, using the learned weights already inspected above. Compare class, score and model identity. Describe what each timer includes and recommend a delivery pattern for the catalog client's arrival pattern and deadline.

## Pipeline extension deliverable

Prepare the unfamiliar partner intake through the documented exercise script. Its active threshold is `24`. Preserve originals, record accepted/rejected decisions, change the active source/test policy and verify actual delivery. Keep the original baseline evidence as history.

In [ ]:
# Keep your intake decisions and evidence references after attempting the assessment.

## Summary

Retain the baseline recipe/weights, raw validation predictions, source/test edits, actual batch/HTTP outputs and recommendation. Later model comparisons confirm the same baseline recipe under the frozen audited source.